In [ ]:
# Install dependencies (required in Colab)
# Colab includes packages such as gradio and google-genai with their own
# pydantic/httpx constraints. Install compatible shared deps first, then
# install the SDK without letting older SDK metadata downgrade them.
%pip install -q --upgrade "pydantic>=2.0,<=2.12.3" "httpx>=0.28.1,<1.0" "attrs>=21.3.0" "python-dateutil>=2.8.0" "click>=8.0.0"
%pip install -q --upgrade --no-deps "docu-devs-api-client>=1.11.0"

In [ ]:
import os
import json
from importlib.metadata import version as package_version
from pathlib import Path

REQUIRED_SDK = (1, 11, 0)
installed_sdk = package_version("docu-devs-api-client")
if tuple(int(part) for part in installed_sdk.split(".")[:3]) < REQUIRED_SDK:
    raise RuntimeError(
        f"docu-devs-api-client {installed_sdk} is older than 1.11.0, which adds case records. "
        "Run the install cell again, or `pip install -U \"docu-devs-api-client>=1.11.0\"`, "
        "then restart the runtime."
    )
print(f"docu-devs-api-client {installed_sdk}")

from docudevs import DocuDevsClient

# Set your API key:
# Option 1: Colab Secrets (recommended) - add DOCUDEVS_API_KEY in the 🔑 sidebar
# Option 2: Environment variable
# Option 3: Set directly below
try:
    from google.colab import userdata
    API_KEY = userdata.get("DOCUDEVS_API_KEY")
except Exception:
    API_KEY = os.getenv("DOCUDEVS_API_KEY", "your-api-key-here")

if not API_KEY or API_KEY == "your-api-key-here":
    raise ValueError("Set your DOCUDEVS_API_KEY - get one at https://docudevs.ai")

client = DocuDevsClient(token=API_KEY)
print("Connected to DocuDevs API")

# ---- Size and data knobs (kept small for the MINI tier) ----
OFF_CATEGORIES = [
    "en:dairies",
    "en:sweet-snacks",
    "en:cereals-and-potatoes",
    "en:beverages",
    "en:frozen-foods",
    "en:sauces",
]
RG_SAMPLE = 60                 # row groups of the export to read (about 1.3% of its rows)
SAMPLE_SEED = 20261011
MIN_LEAF_PRODUCTS = 10         # categories with fewer products are rolled up to their parent
RECORDS_TARGET = 2500          # records in the knowledge base
HOLDOUT_TARGET = 100           # held-out products that get classified
REVIEW_BATCH = 30              # records classified in the review step
TARGET_DEPTH = [1, 2, 3]       # tree levels reported with each result
LLM_LEVEL = "MINI"
CLASSIFY_BUDGET_TOKENS = 600_000
HOLDOUT_EVAL_SAMPLE = 100
RUN_AGENT = False              # True starts the optional headless investigation
DELETE_CASE_AT_END = True

DATA_DIR = Path("off-cache")   # local cache of the taxonomy and the sample; never committed
DATA_DIR.mkdir(exist_ok=True)

In [ ]:
import bisect
import io
import random
import time
import urllib.error
import urllib.request
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor

import pyarrow.parquet as pq

HF_EXPORT_URL = "https://huggingface.co/datasets/openfoodfacts/product-database/resolve/main/food.parquet"
OFF_TAXONOMY_URL = "https://static.openfoodfacts.org/data/taxonomies/categories.json"
USER_AGENT = "docudevs-python-examples/1.0 (+https://github.com/docudevs/python-examples)"
NEEDED_COLUMNS = [
    "code",
    "product_name",
    "generic_name",
    "brands",
    "quantity",
    "ingredients_text",
    "categories_tags",
    "countries_tags",
]
TEXT_COLUMNS = ("name", "generic_name", "brand", "quantity", "ingredients")
FIELD_MAPPING = {
    "key": "ean",
    "text": {column: column for column in TEXT_COLUMNS},
    "label": "leaf",
    "metadata": {"brand": "brand"},
}
EMBEDDED_TEXT = {
    "template": "{name}\n{generic_name}\n{brand}\n{quantity}\n{ingredients}",
    "skipEmptyFields": True,
}


class PrefetchedRangeFile(io.RawIOBase):
    """Read-only file over HTTP range requests. Byte spans are downloaded in parallel first."""

    def __init__(self, url):
        self.url = url
        self.size = self._head_size()
        self.pos = 0
        self.bytes_downloaded = 0
        self._starts = []
        self._spans = []

    def _head_size(self):
        request = urllib.request.Request(self.url, method="HEAD", headers={"User-Agent": USER_AGENT})
        with urllib.request.urlopen(request, timeout=120) as response:
            return int(response.headers["Content-Length"])

    def readable(self):
        return True

    def seekable(self):
        return True

    def tell(self):
        return self.pos

    def seek(self, offset, whence=0):
        self.pos = {0: 0, 1: self.pos, 2: self.size}[whence] + offset
        return self.pos

    def _fetch(self, start, end, attempts=3):
        request = urllib.request.Request(
            self.url, headers={"Range": f"bytes={start}-{end - 1}", "User-Agent": USER_AGENT}
        )
        for attempt in range(1, attempts + 1):
            try:
                with urllib.request.urlopen(request, timeout=300) as response:
                    data = response.read()
                break
            except urllib.error.URLError:
                if attempt == attempts:
                    raise
                time.sleep(2 * attempt)
        assert len(data) == end - start, (start, end, len(data))
        self.bytes_downloaded += len(data)
        return data

    def prefetch(self, spans, workers=16):
        merged = []
        for start, end in sorted(spans):
            if merged and start <= merged[-1][1]:
                merged[-1][1] = max(merged[-1][1], end)
            else:
                merged.append([start, end])
        with ThreadPoolExecutor(max_workers=workers) as pool:
            fetched = pool.map(lambda span: self._fetch(*span), [tuple(span) for span in merged])
            for (start, _), data in zip(merged, fetched):
                self._spans.append((start, data))
        self._spans.sort(key=lambda item: item[0])
        self._starts = [start for start, _ in self._spans]

    def _lookup(self, start, end):
        index = bisect.bisect_right(self._starts, start) - 1
        if index >= 0:
            span_start, data = self._spans[index]
            if end <= span_start + len(data):
                return data[start - span_start:end - span_start]
        return self._fetch(start, end)

    def read(self, size=-1):
        if size is None or size < 0:
            size = self.size - self.pos
        end = min(self.pos + size, self.size)
        if end <= self.pos:
            return b""
        data = self._lookup(self.pos, end)
        self.pos = end
        return data

    def readinto(self, buffer):
        data = self.read(len(buffer))
        buffer[: len(data)] = data
        return len(data)


def column_spans(metadata, row_group, names):
    """Byte spans of the column chunks of one row group whose top-level column is in names."""
    group = metadata.row_group(row_group)
    spans = []
    for index in range(group.num_columns):
        column = group.column(index)
        if column.path_in_schema.split(".")[0] in names:
            start = column.dictionary_page_offset if column.has_dictionary_page else column.data_page_offset
            spans.append((start, start + column.total_compressed_size))
    return spans


def load_sample(cache_file):
    """Return the rows of RG_SAMPLE random row groups, reading only NEEDED_COLUMNS."""
    if cache_file.exists():
        return pq.read_table(cache_file).to_pylist(), 0
    handle = PrefetchedRangeFile(HF_EXPORT_URL)
    metadata = pq.read_metadata(handle)
    row_groups = sorted(random.Random(SAMPLE_SEED).sample(range(metadata.num_row_groups), RG_SAMPLE))
    spans = [span for rg in row_groups for span in column_spans(metadata, rg, set(NEEDED_COLUMNS))]
    handle.prefetch(spans)
    table = pq.ParquetFile(handle, metadata=metadata).read_row_groups(
        row_groups, columns=NEEDED_COLUMNS, use_threads=False
    )
    pq.write_table(table, cache_file)
    return table.to_pylist(), handle.bytes_downloaded


def load_taxonomy(cache_file):
    if not cache_file.exists():
        request = urllib.request.Request(OFF_TAXONOMY_URL, headers={"User-Agent": USER_AGENT})
        with urllib.request.urlopen(request, timeout=300) as response:
            cache_file.write_bytes(response.read())
    return json.loads(cache_file.read_text(encoding="utf-8"))


def single_chain(tag, parents):
    """Return [root, ..., tag] if every step up has exactly one parent, otherwise None."""
    if tag not in parents:
        return None
    chain = [tag]
    while parents[chain[-1]]:
        if len(parents[chain[-1]]) > 1:
            return None
        parent = parents[chain[-1]][0]
        if parent in chain or parent not in parents:
            return None
        chain.append(parent)
    return list(reversed(chain))


def most_specific_chain(tags, parents, popularity):
    """The deepest category with an unambiguous path to a root; ties go to the more common one."""
    chains = [chain for chain in (single_chain(tag, parents) for tag in tags) if chain]
    if not chains:
        return None
    return max(chains, key=lambda chain: (len(chain), popularity.get(chain[-1], 0), chain[-1]))


def parents_from_taxonomy(taxonomy):
    return {tag: list(entry.get("parents") or []) for tag, entry in taxonomy.items()}


def category_name(tag, taxonomy):
    names = (taxonomy.get(tag) or {}).get("name") or {}
    return names.get("en") or tag.split(":", 1)[-1].replace("-", " ").capitalize()


def parent_code(code):
    return code.rsplit(".", 1)[0] if "." in code else ""


def prefix(code, depth):
    return ".".join(code.split(".")[:depth])


def pick_text(entries):
    entries = entries or []
    for entry in entries:
        if entry.get("lang") == "en" and entry.get("text"):
            return entry["text"].strip()
    for entry in entries:
        if entry.get("text"):
            return entry["text"].strip()
    return ""


def normalise_product(raw):
    code = str(raw.get("code") or "").strip()
    name = pick_text(raw.get("product_name"))[:200]
    if not code or not name:
        return None
    return {
        "ean": code,
        "name": name,
        "generic_name": pick_text(raw.get("generic_name"))[:200],
        "brand": (raw.get("brands") or "").split(",")[0].strip()[:100],
        "quantity": (raw.get("quantity") or "").strip()[:100],
        "ingredients": pick_text(raw.get("ingredients_text"))[:300],
        "categories_tags": list(raw.get("categories_tags") or []),
        "countries_tags": list(raw.get("countries_tags") or []),
    }


def unique_products(raw_products):
    products = {}
    for raw in raw_products:
        product = normalise_product(raw)
        if product and product["ean"] not in products:
            products[product["ean"]] = product
    return list(products.values())


def chains_by_ean(products, parents, popularity):
    """Leaf chains of the products. A product whose deepest category is a root is not labelled."""
    chains = {}
    for product in products:
        chain = most_specific_chain(product["categories_tags"], parents, popularity)
        if chain and len(chain) > 1:
            chains[product["ean"]] = chain
    return chains


def roll_up(chains, min_products):
    """Move products of a category with fewer than min_products direct products to its parent."""
    labels = dict(chains)
    deepest = max((len(chain) for chain in labels.values()), default=0)
    for depth in range(deepest, 0, -1):
        counts = Counter(chain[-1] for chain in labels.values() if len(chain) == depth)
        small = {tag for tag, count in counts.items() if count < min_products}
        for ean, chain in list(labels.items()):
            if len(chain) == depth and chain[-1] in small:
                labels[ean] = chain[:-1]
    return {ean: chain for ean, chain in labels.items() if chain}


def prune_to_leaves(chains, min_products):
    """Drop products until every remaining label is a leaf with enough products.

    A category with fewer than min_products products is removed, and so is a product whose
    category is an ancestor of another kept category (it would be a non-leaf label).
    Repeat until nothing changes, because each removal can expose another.
    """
    kept = dict(chains)
    while True:
        counts = Counter(chain[-1] for chain in kept.values())
        ancestors = {tag for chain in kept.values() for tag in chain[:-1]}
        dropped = [
            ean
            for ean, chain in kept.items()
            if counts[chain[-1]] < min_products or chain[-1] in ancestors
        ]
        if not dropped:
            return kept
        for ean in dropped:
            del kept[ean]


def build_tree(chains, taxonomy):
    """Build dotted-code tree nodes from leaf chains. Returns (nodes, code_of_tag)."""
    parent_of = {}
    for chain in chains.values():
        for parent, child in zip(chain, chain[1:]):
            parent_of[child] = parent
    tags = {tag for chain in chains.values() for tag in chain}
    children = defaultdict(list)
    for child, parent in parent_of.items():
        children[parent].append(child)
    roots = sorted(
        (tag for tag in tags if tag not in parent_of),
        key=lambda tag: category_name(tag, taxonomy),
    )
    code_of = {}
    nodes = []
    queue = [(tag, str(index), None) for index, tag in enumerate(roots, start=1)]
    while queue:
        tag, code, parent = queue.pop(0)
        code_of[tag] = code
        nodes.append({"code": code, "parent": parent, "name": category_name(tag, taxonomy)})
        kids = sorted(children[tag], key=lambda kid: category_name(kid, taxonomy))
        queue.extend((kid, f"{code}.{index}", code) for index, kid in enumerate(kids, start=1))
    return nodes, code_of


def group_key(product):
    if product["brand"]:
        return "brand:" + product["brand"].lower()
    words = product["name"].lower().split()
    return "name:" + (words[0] if words else product["ean"])


def split_records_and_holdout(products, records_target, holdout_target, seed):
    """Move whole brand groups (or name-word groups) to the holdout, so no group is on both sides."""
    groups = defaultdict(list)
    for product in products:
        groups[group_key(product)].append(product)
    keys = sorted(groups)
    random.Random(seed).shuffle(keys)
    held = set()
    holdout = []
    for key in keys:
        if len(holdout) >= holdout_target:
            break
        if len(holdout) + len(groups[key]) <= holdout_target:
            held.add(key)
            holdout.extend(groups[key])
    rest = [product for key in sorted(groups) if key not in held for product in groups[key]]
    random.Random(seed + 1).shuffle(rest)
    return rest[:records_target], holdout


def record_row(product, leaf_code):
    row = {column: product[column] for column in TEXT_COLUMNS}
    row.update({"ean": product["ean"], "leaf": leaf_code})
    return row


def inline_row(product):
    row = {"key": product["ean"], "text": {c: product[c] for c in TEXT_COLUMNS if product[c]}}
    if product["brand"]:
        row["metadata"] = {"brand": product["brand"]}
    return row


def accuracy_by_level(results, truth_by_key, levels):
    scores = {}
    for depth in levels:
        right = 0
        for row in results:
            chosen = (row.get("chosen") or {}).get("code")
            if chosen and prefix(chosen, depth) == prefix(truth_by_key[row["key"]], depth):
                right += 1
        scores[depth] = (right, len(results))
    return scores


def alternative_leaf(code, leaves):
    siblings = [leaf for leaf in leaves if leaf != code and parent_code(leaf) == parent_code(code)]
    others = [leaf for leaf in leaves if leaf != code]
    return (siblings or others)[0]

In [ ]:
taxonomy = load_taxonomy(DATA_DIR / "categories.json")
parents = parents_from_taxonomy(taxonomy)
print(f"Taxonomy: {len(parents):,} categories")

sample_file = DATA_DIR / f"sample-rg{RG_SAMPLE}-seed{SAMPLE_SEED}.parquet"
sample_rows, downloaded_bytes = load_sample(sample_file)
if downloaded_bytes:
    print(f"Downloaded {downloaded_bytes / 1e6:.1f} MB of column data from the Parquet export")
else:
    print(f"Loaded the cached sample {sample_file}")

wanted = set(OFF_CATEGORIES)
matched = [row for row in sample_rows if wanted & set(row["categories_tags"] or [])]
products = unique_products(matched)
print(f"Sample: {len(sample_rows):,} products in {RG_SAMPLE} row groups")
print(f"In the chosen categories: {len(matched):,} rows, {len(products):,} unique products")

In [ ]:
finland = sum(1 for product in products if "en:finland" in product["countries_tags"])
print(f"Sold in Finland: {finland:,} products")

popularity = Counter(tag for product in products for tag in set(product["categories_tags"]))
chains = chains_by_ean(products, parents, popularity)
print(f"Products with a deepest category below a root: {len(chains):,}")
depths = Counter(len(chain) for chain in chains.values())
print("Depth of each product's most specific category (levels):")
for depth in sorted(depths):
    print(f"  {depth}: {depths[depth]:,}")

print("\nExamples:")
for product in products[:3]:
    chain = chains.get(product["ean"])
    path = " > ".join(category_name(tag, taxonomy) for tag in chain) if chain else "(no sub-category)"
    print(f"- {product['name']} | brand: {product['brand'] or '-'} | {product['quantity'] or '-'}")
    print(f"  {path}")

In [ ]:
rolled = roll_up(chains, MIN_LEAF_PRODUCTS)
kept_chains = {ean: chain for ean, chain in prune_to_leaves(rolled, MIN_LEAF_PRODUCTS).items() if len(chain) > 1}
nodes, code_of = build_tree(kept_chains, taxonomy)
leaves = sorted({code_of[chain[-1]] for chain in kept_chains.values()})
leaf_of = {ean: code_of[chain[-1]] for ean, chain in kept_chains.items()}
max_depth = max(len(code.split(".")) for code in leaves)
leaf_sizes = Counter(leaf_of.values())
print(f"Rolled up: {len(chains) - len(rolled):,} products moved to a parent category")
print(f"Pruned: {len(rolled) - len(kept_chains):,} products dropped to keep every label a leaf")
print(f"Tree: {len(nodes)} categories, {len(leaves)} leaves, depth {max_depth}")
print(f"Products kept: {len(kept_chains):,}")
print("Largest leaves:")
for code, count in leaf_sizes.most_common(5):
    name = next(node["name"] for node in nodes if node["code"] == code)
    print(f"  {code:10} {name} ({count})")

In [ ]:
kept_products = [product for product in products if product["ean"] in kept_chains]
records_pool, holdout_pool = split_records_and_holdout(
    kept_products,
    RECORDS_TARGET,
    HOLDOUT_TARGET,
    seed=SAMPLE_SEED,
)
records_leaves = {leaf_of[product["ean"]] for product in records_pool}
holdout = [product for product in holdout_pool if leaf_of[product["ean"]] in records_leaves]

records_rows = [record_row(product, leaf_of[product["ean"]]) for product in records_pool]
truth_by_key = {product["ean"]: leaf_of[product["ean"]] for product in holdout}
records_by_ean = {row["ean"]: row for row in records_rows}
print(f"Records: {len(records_rows):,} products in {len(records_leaves)} leaves")
print(f"Holdout: {len(holdout)} products from brands that are not in the records")

In [ ]:
def level_fanout(nodes):
    children = defaultdict(int)
    for node in nodes:
        if node["parent"]:
            children[node["parent"]] += 1
    rows = []
    for depth in range(1, max_depth + 1):
        at_depth = [node for node in nodes if len(node["code"].split(".")) == depth]
        leaves_here = [node for node in at_depth if node["code"] in leaves]
        internal = [node for node in at_depth if node["code"] not in leaves]
        mean = sum(children[node["code"]] for node in internal) / len(internal) if internal else 0
        rows.append((depth, len(at_depth), len(leaves_here), round(mean, 1)))
    return rows


print("Level  categories  leaves  mean children (internal nodes)")
for depth, total, leaf_count, mean in level_fanout(nodes):
    print(f"{depth:>5}  {total:>10}  {leaf_count:>6}  {mean:>10}")
leaf_counts = sorted(leaf_sizes.values())
print(f"Products per leaf: min {leaf_counts[0]}, median {leaf_counts[len(leaf_counts) // 2]}, max {leaf_counts[-1]}")

In [ ]:
from docudevs.models.cases_controller_create_case_request import CasesControllerCreateCaseRequest


def get_case_id(response):
    """Extract the case ID from a 200 or 201 response."""
    if response.parsed is not None:
        return response.parsed.id
    return json.loads(response.content.decode())["id"]


created = await client.create_case(
    body=CasesControllerCreateCaseRequest(name="Open Food Facts products (notebook)")
)
if created.status_code not in (200, 201):
    raise RuntimeError(f"Error creating case: {created.status_code} {created.content.decode()}")
case_id = get_case_id(created)
print(f"Created case {case_id}")

dataset = await client.set_dataset(
    case_id,
    FIELD_MAPPING,
    embedded_text=EMBEDDED_TEXT,
    index_settings={"topK": 10, "hybrid": False},
    classify_defaults={"targetDepth": TARGET_DEPTH},
)
print("Dataset configured")

In [ ]:
imported = await client.bulk_upsert_records(case_id, records_rows)
final = await client.wait_for_import(case_id, imported["importId"], timeout=1800, poll_interval=5.0)
if final["status"] != "COMPLETED":
    raise RuntimeError(f"Import ended with {final['status']}")
print({key: final.get(key) for key in ("rowsReceived", "rowsRejected", "recordsAdded")})

In [ ]:
uploaded = await client.upload_tree(case_id, nodes, name="Open Food Facts categories")
version_id = uploaded["treeVersionId"]
await client.activate_tree(case_id, version_id, wait=True, timeout=1800)
print(f"Tree version {version_id} is active with {len(leaves)} leaves")

In [ ]:
classify_config = {
    "llmLevel": LLM_LEVEL,
    "concurrency": 4,
    "maxTotalTokens": CLASSIFY_BUDGET_TOKENS,
    "targetDepth": TARGET_DEPTH,
}
accepted = await client.classify(case_id, rows=[inline_row(p) for p in holdout], config=classify_config)
job_id = int(accepted["jobId"])
job = await client.wait_for_classify(case_id, job_id, timeout=3600, poll_interval=5.0)
print(f"Classify job {job_id}: {job['status']}", job.get("counts"))

results = [row async for row in client.get_classify_results(case_id, job_id)]
declined = sum(1 for row in results if not row.get("chosen"))
print(f"{len(results)} rows, {declined} declined or failed")

levels = sorted(set(TARGET_DEPTH) | {max_depth})
for depth, (right, total) in accuracy_by_level(results, truth_by_key, levels).items():
    print(f"Level {depth}: {right}/{total} correct ({right / total:.0%})")

In [ ]:
flagged = [row for row in results if (row.get("review") or {}).get("required")]
print(f"{len(flagged)} of {len(results)} rows need review")

for row in flagged[:5]:
    chosen = row.get("chosen") or {}
    print(f"\nRow {row['rowId']}: {row['key']} ({truth_by_key[row['key']]} is the historical label)")
    print(f"  Chosen: {' > '.join(chosen.get('path', [])) or 'declined'}")
    print(f"  Reasons: {', '.join(row['review']['reasons'])}")
    if row.get("alternativeBestFit"):
        print(f"  Model's alternative: {row['alternativeBestFit'].get('reason')}")
    for ean in row.get("evidenceKeys", [])[:3]:
        evidence = records_by_ean.get(ean, {})
        print(f"  Evidence: {evidence.get('name', ean)} -> {evidence.get('leaf', '?')}")

In [ ]:
async def wait_for_label_write(review):
    write = review.get("labelWrite") or {}
    if write.get("importId") is not None:
        final = await client.wait_for_import(case_id, write["importId"], timeout=1800, poll_interval=2.0)
        if final["status"] != "COMPLETED":
            raise RuntimeError(f"Label write ended {final['status']}")


review_keys = [row["ean"] for row in random.Random(SAMPLE_SEED + 2).sample(records_rows, REVIEW_BATCH)]
review_accepted = await client.classify(case_id, keys=review_keys, config=classify_config)
review_job_id = int(review_accepted["jobId"])
await client.wait_for_classify(case_id, review_job_id, timeout=3600, poll_interval=5.0)
review_results = [row async for row in client.get_classify_results(case_id, review_job_id)]

right = [row for row in review_results if row.get("chosen") and row["chosen"]["code"] == records_by_ean[row["key"]]["leaf"]]
wrong = [row for row in review_results if row.get("chosen") and row["chosen"]["code"] != records_by_ean[row["key"]]["leaf"]]
print(f"Review batch: {len(right)} agree with the record label, {len(wrong)} disagree")

accept_row = right[0]
accepted_review = await client.review_classify(case_id, review_job_id, accept_row["rowId"], "accept")
await wait_for_label_write(accepted_review)
accepted_label = (await client.get_record(case_id, accept_row["key"])).get("label")
print(f"Accepted {accept_row['key']}: label is {accepted_label}")

correct_row = wrong[0]
historical_code = records_by_ean[correct_row["key"]]["leaf"]
corrected_review = await client.review_classify(
    case_id,
    review_job_id,
    correct_row["rowId"],
    "correct",
    code=historical_code,
    reason="Reviewed: the product name and ingredients describe the historical category, not the one the model chose.",
)
await wait_for_label_write(corrected_review)
corrected_label = (await client.get_record(case_id, correct_row["key"])).get("label")
print(f"Corrected {correct_row['key']}: model said {correct_row['chosen']['code']}, review label is {corrected_label}")

In [ ]:
truth_code = historical_code
predicted_code = correct_row["chosen"]["code"]
truth_name = next(node["name"] for node in nodes if node["code"] == truth_code)
predicted_name = next(node["name"] for node in nodes if node["code"] == predicted_code)

rule_text = (
    f"Between '{truth_name}' and '{predicted_name}': file a product under '{truth_name}' "
    f"when its name or generic name describes that product type. Use '{predicted_name}' only "
    "when the product name names that type."
)
entry = await client.add_memo_entry(
    case_id,
    "instruction",
    {"kind": "subtree", "code": parent_code(truth_code) or truth_code},
    rule_text,
)
approved = await client.approve_memo_entry(case_id, entry["id"])
print(f"Approved memo entry {entry['id']} (status {approved.get('status')})")
print(rule_text)

In [ ]:
variants = [
    {
        "name": "without-memo",
        "config": {"llmLevel": LLM_LEVEL, "targetDepth": TARGET_DEPTH, "memo": {"use": "none"}},
    },
    {"name": "with-memo"},
]
started = await client.start_holdout_eval(
    case_id,
    variants,
    sample_size=HOLDOUT_EVAL_SAMPLE,
    seed=7,
    config={"llmLevel": LLM_LEVEL, "targetDepth": TARGET_DEPTH},
)
finished = await client.wait_for_eval(case_id, started["evalId"], timeout=3600, poll_interval=5.0)

report = finished.get("report") or {}
print("Grouping:", (report.get("sample") or {}).get("grouping"))
for variant in report.get("variants") or []:
    target = ((variant.get("score") or {}).get("targetAccuracy")) or {}
    print(f"{variant.get('name')}: target accuracy {target.get('all')} over {target.get('nAll')} rows")
print("Paired differences:")
print(json.dumps(report.get("comparisons"), indent=2)[:2000])

In [ ]:
if RUN_AGENT:
    started_run = await client.start_agent_run(
        case_id,
        "investigate-flagged",
        params={"job_id": job_id, "max_rows": 3},
        budget={"max_total_tokens": 20_000},
        origin_job_id=job_id,
    )
    run = await client.wait_for_agent_run(case_id, started_run["runId"], timeout=1800, poll_interval=5.0)
    print(run["status"], "| tokens:", run.get("tokensUsed"), "| note:", run.get("error"))
    print((run.get("report") or {}).get("summary"))
    for action in (await client.list_agent_actions(case_id)).get("items", []):
        print(action.get("status"), "-", action.get("summary"))
else:
    print("Set RUN_AGENT = True to start a headless investigation (20k token budget).")

In [ ]:
if DELETE_CASE_AT_END:
    await client.delete_case(case_id)
    print(f"Deleted case {case_id}")
else:
    print(f"Kept case {case_id}")